# Format data from different sites

In [24]:
import pandas as pd
import numpy as np

## Chile

In [25]:
acr2int = {'A': 1, 'B': 2, 'C': 3, 'D': 4}

In [26]:
data = pd.read_csv("data/chile/complete_DH_data.csv")

In [27]:
data = data.rename(columns={
    'exam_date': 'Exam Date', 
    'patient_id': 'Patient ID',
    'age': 'Age at Exam',
    'acr_density': 'Density',
    'cancer': 'Cancer (YES | NO)',
    'event_time': 'Date of Cancer Diagnosis'
})
data['Density'] = data['Density'].map(acr2int)
data["Exam ID"] = data["Patient ID"].astype(str) + '_' + data["Exam Date"].astype(str)
data["Date of Last Negative Mammogram"] = data['Date of Cancer Diagnosis']
data["Ethnicity"] = ["Chilean"] * len(data)
del data['Unnamed: 0']

In [ ]:
len(data["Exam ID"]), len(set(data["Exam ID"]))

(445122, 440903)

(445122, 440903)

In [ ]:
data['Density'].min(), data['Density'].max()

(1, 4)

In [ ]:
for col in [
    "Exam Date",
    "Date of Last Negative Mammogram",
    "Date of Cancer Diagnosis",
]:
    data[col] = pd.to_datetime(data[col])  #'%m/%d/%Y' # , format='%Y-%m-%d'


In [ ]:
data.to_csv("data/chile/formatted_complete_DH_data.csv", index=False)

## UCSF

In [ ]:
data = pd.read_csv("data/UCSF/anonymized_risk_model_preds_w_outcomes_jan_5_2025.csv")
data = data[ ['years_to_cancer', 'years_to_last_followup', 'StudyDate', 'age', 'density_human', 'anonymized_pid', 'anonymized_exam_id']]

In [ ]:
len(data), len(set(data['anonymized_exam_id']))

(191333, 130712)

In [ ]:
data = data.rename(columns={
    'StudyDate': 'Exam Date', 
    'anonymized_pid': 'Patient ID',
    'anonymized_exam_id': 'Exam ID',
    'age': 'Age at Exam',
    'density_human': 'Density',
})
data['Exam Date'] = pd.to_datetime(data['Exam Date'], format="%Y") 
data["Cancer (YES | NO)"] = data["years_to_cancer"] < 100
data["Date of Last Negative Mammogram"] = data.apply(
    lambda row: row['Exam Date'] + pd.DateOffset(years=row['years_to_last_followup']), axis=1
)
data["Date of Cancer Diagnosis"] = data.apply(
    lambda row: row['Exam Date'] + pd.DateOffset(years=row['years_to_cancer']), axis=1
)
data["Ethnicity"] = ["UCSF"] * len(data)
data = data.dropna(subset=['Density'])

In [ ]:
data['Density'].min(), data['Density'].max()

(0.0, 3.0)

In [ ]:
data['Density'] = data['Density'] + 1

In [ ]:
data['Density'].min(), data['Density'].max()

(1.0, 4.0)

In [ ]:
data.to_csv("data/UCSF/formatted_anonymized_risk_model_preds_w_outcomes_jan_5_2025.csv", index=False)

## CMU - Taiwan

In [ ]:
data = pd.read_excel("data/taiwan-cmu/input_ori_20241127.xlsx")

In [ ]:
data['Density'].min(), data['Density'].max()

(1, 4)

In [ ]:
len(data["Exam ID"]), len(set(data["Exam ID"]))

(54967, 54967)

## CGMH - Taiwan

In [ ]:
data_outcomes = pd.read_excel('data/taiwan-cgmh/Mammographic Density Evaluator CGMH 20241211.xlsx')
data = pd.read_excel("data/taiwan-cgmh/MIRAI20250110 to Peter.xlsx")
# data_age = pd.read_excel("data/taiwan-cgmh/age_20250106.xlsx")b

In [ ]:
len(data['exam_id']), len(set(data['exam_id']))

(13175, 13175)

In [ ]:
data = data[['exam_id', 'StudyDate', 'AccessionNumber', 'CGMH Report Breast density', 'Age']]
data_outcomes = data_outcomes[['exam_id', 'StudyDate', 'date_of_cancer', 'date_of_last_screening_followup ','death_date']]

In [ ]:
data = pd.merge(data, data_outcomes, on='exam_id', how='outer')

In [ ]:
data = data.rename(columns={
    'exam_id': 'Exam ID',
    'StudyDate_x': 'Exam Date', 
    'AccessionNumber': 'Patient ID',
    'CGMH Report Breast density': 'Density',
    'date_of_last_screening_followup ': 'Date of Last Negative Mammogram',
    'date_of_cancer': 'Date of Cancer Diagnosis',
    'Age': 'Age at Exam'
})
data["Cancer (YES | NO)"] = ~data['Date of Cancer Diagnosis'].isna()
del data['StudyDate_y']
del data['death_date']

In [ ]:
data = data.dropna(subset=["Date of Last Negative Mammogram", "Date of Cancer Diagnosis"], how='all')

In [ ]:
data["Exam Date"] = pd.to_datetime(data["Exam Date"].astype(int).astype(str))
data["Date of Last Negative Mammogram"] = pd.to_datetime(data["Date of Last Negative Mammogram"].astype(int).astype(str))

In [ ]:
cancer_dates = []
for _, row in data.iterrows():
    if np.isnan(row['Date of Cancer Diagnosis']):
        cancer_dates.append( row['Exam Date'] + pd.DateOffset(years=100) )
    else:
        try:
            cancer_dates.append( pd.to_datetime(str(int(row['Date of Cancer Diagnosis']))) )
        except ValueError:
            cancer_dates.append(None)
data['Date of Cancer Diagnosis'] = cancer_dates
data = data.dropna(subset=['Date of Cancer Diagnosis'])

In [ ]:
data['Density'].min(), data['Density'].max()

(1, 4)

In [ ]:
data.to_csv("data/taiwan-cgmh/formatted_MIRAI20250110 to Peter.csv", index=False)

## Apollo

In [ ]:
data_outcomes = pd.read_csv('data/apollo/data.csv')

In [ ]:
len(data_outcomes['Patient ID']), len(set(data_outcomes['Patient ID']))

(7876, 3177)

In [ ]:
for col in [
    "Exam Date",
    "Date of Last Negative Mammogram",
    "Date of Cancer Diagnosis",
]:
    data_outcomes[col] = pd.to_datetime(data_outcomes[col]) 

In [ ]:
# fix cancer label: if cancer and date of diagnosis is after exam
data_outcomes['Cancer (YES | NO) OLD'] = data_outcomes['Cancer (YES | NO)']
data_outcomes['Cancer (YES | NO)'] = data_outcomes.apply( lambda row: row['Cancer (YES | NO)'] and row['Date of Cancer Diagnosis'] > row['Exam Date'], axis=1)

In [ ]:
# cases where diagnosis is before exam, assuming cancer was treated then and is not present at exam
pid2lastnegativedate = {}
for i, row in data_outcomes.iterrows():
    if row['Date of Cancer Diagnosis'] < row['Exam Date']:
        last_negative = pid2lastnegativedate.get(row['Patient ID'], None) 
        if last_negative is None or last_negative < row['Exam Date']:
            pid2lastnegativedate[row['Patient ID']] = row['Exam Date']

last_negative_dates = []
for i, row in data_outcomes.iterrows():
    if row['Patient ID'] in pid2lastnegativedate:
        last_negative_dates.append(pid2lastnegativedate[row['Patient ID']])
    else:
        last_negative_dates.append(row['Date of Last Negative Mammogram'])
    

In [ ]:
data_outcomes['Date of Last Negative Mammogram OLD'] = data_outcomes['Date of Last Negative Mammogram']
data_outcomes['Date of Last Negative Mammogram'] = last_negative_dates
data_outcomes["Prior Cancer"] = data_outcomes["Date of Cancer Diagnosis"] < data_outcomes["Exam Date"]

In [ ]:

data_outcomes[~data_outcomes['Prior Cancer']].to_csv("data/apollo/formatted_data_no_prior_cancer.csv", index=False)
data_outcomes.to_csv("data/apollo/formatted_data.csv", index=False)

In [ ]:
data_outcomes = pd.read_csv('data/apollo/mammo_density_7984_rawdata.csv') 

In [ ]:
len(data_outcomes['Patient ID']), len(set(data_outcomes['Patient ID']))

(18770, 7984)

In [ ]:
for col in [
    "Exam Date",
    "Date of Last Negative Mammogram",
    "Date of Cancer Diagnosis",
]:
    data_outcomes[col] = pd.to_datetime(data_outcomes[col]) 
    

In [ ]:
# fix cancer label: if cancer and date of diagnosis is after exam
data_outcomes['Cancer (YES | NO) OLD'] = data_outcomes['Cancer (YES | NO)']
data_outcomes['Cancer (YES | NO)'] = data_outcomes.apply( lambda row: row['Cancer (YES | NO)'] and row['Date of Cancer Diagnosis'] > row['Exam Date'], axis=1)

In [ ]:
# cases where diagnosis is before exam, assuming cancer was treated then and is not present at exam
pid2lastnegativedate = {}
for i, row in data_outcomes.iterrows():
    if row['Date of Cancer Diagnosis'] < row['Exam Date']:
        last_negative = pid2lastnegativedate.get(row['Patient ID'], None) 
        if last_negative is None or last_negative < row['Exam Date']:
            pid2lastnegativedate[row['Patient ID']] = row['Exam Date']

last_negative_dates = []
for i, row in data_outcomes.iterrows():
    if row['Patient ID'] in pid2lastnegativedate:
        last_negative_dates.append(pid2lastnegativedate[row['Patient ID']])
    else:
        last_negative_dates.append(row['Date of Last Negative Mammogram'])
    

In [ ]:
data_outcomes['Date of Last Negative Mammogram OLD'] = data_outcomes['Date of Last Negative Mammogram']
data_outcomes['Date of Last Negative Mammogram'] = last_negative_dates
data_outcomes["Prior Cancer"] = data_outcomes["Date of Cancer Diagnosis"] < data_outcomes["Exam Date"]

In [ ]:

data_outcomes[~data_outcomes['Prior Cancer']].to_csv("data/apollo/formatted_data_no_prior_cancer.csv", index=False)
data_outcomes.to_csv("data/apollo/formatted_data.csv", index=False)

In [ ]:
len(data_outcomes['Patient ID']), len(set(data_outcomes['Patient ID']))


(18770, 7984)

## Emory

In [ ]:
data = pd.read_csv('data/emory/input_file_one_outcome.csv')

In [ ]:
len(data["Exam ID"]), len(set(data["Exam ID"]))

(348527, 348527)

In [ ]:
data['Density'].min(), data['Density'].max()

(1.0, 4.0)

In [ ]:
del data["year"]

## Colombia

In [ ]:
data = pd.read_csv("data/colombia/breast_density_project_db.csv")
data['diagnosed_after_exam'] = (data['Date of Cancer Diagnosis'] >= data['Exam Date']) | (data['Date of Last Negative Mammogram'] >= data['Exam Date'])
data = data[data['diagnosed_after_exam'] == True]
data["Cancer (YES | NO)"] = (data["Cancer (YES | NO)"] == "YES")*1
len(data["Exam ID"]), len(set(data["Exam ID"])), len(set(data["Patient ID"]))
data.to_csv("data/colombia/formatted_breast_density_project_db.csv", index=False)


## Combined

In [ ]:
data1 = pd.read_csv("data/chile/formatted_complete_DH_data.csv")
data2 = pd.read_csv("data/UCSF/formatted_anonymized_risk_model_preds_w_outcomes_jan_5_2025.csv")
data3 = pd.read_excel("data/taiwan-cmu/input_ori_20241127.xlsx")
data4 = pd.read_csv("data/taiwan-cgmh/formatted_MIRAI20250110 to Peter.csv")
data5 = pd.read_csv("data/apollo/formatted_data.csv")
data6 = pd.read_csv('data/emory/input_file_one_outcome.csv')
data7 = pd.read_csv('data/colombia/formatted_breast_density_project_db.csv')

In [ ]:
data_list = [
    (data1, "Chile"), 
    (data2, "UCSF"), 
    (data3, "Taiwan-CMU"), 
    (data4, "Taiwan-CGMH"), 
    (data5, "Apollo"),
    (data6, "Emory"),
    (data7, "Colombia"),
]

In [ ]:
for data, name in data_list:
    data["SITE"] = name
    for col in [
        "Exam Date",
        "Date of Last Negative Mammogram",
        "Date of Cancer Diagnosis",
    ]:
        data[col] = pd.to_datetime(data[col])

In [ ]:
# harmonize ethnicity column

In [ ]:
data1['Ethnicity'] = "hispanic"
data2['Ethnicity'] = "unk"
data3['Ethnicity'] = "asian"
data4['Ethnicity'] = "asian"
data5['Ethnicity'] = "indian"
data6['Ethnicity'] = data6['Ethnicity'].str.lower()
data7.loc[ data7['Ethnicity'] == 'Afroamericano', 'Ethnicity'] = "black"
data7.loc[ data7['Ethnicity'] ==  'Caucásico', 'Ethnicity'] = "white"
data7.loc[ data7['Ethnicity'] ==  '-', 'Ethnicity'] = "hispanic"

In [ ]:
result = pd.concat([data1, data2, data3, data4, data5, data6, data7], ignore_index=True)

In [ ]:
result.to_csv("data/merged_data.csv", index=False)

In [53]:
data1 = pd.read_csv("data/chile/formatted_complete_DH_data.csv")
data2 = pd.read_csv("data/UCSF/formatted_anonymized_risk_model_preds_w_outcomes_jan_5_2025.csv")
# data3 = pd.read_excel("data/taiwan-cmu/input_ori_20241127.xlsx")
# data4 = pd.read_csv("data/taiwan-cgmh/formatted_MIRAI20250110 to Peter.csv")
data3 = pd.read_excel("data/taiwan-cmu/input_ori_20250808.xlsx")
data4 = pd.read_csv("data/taiwan-cgmh/formatted_minus_asymptomatic_08122025.csv")
data5 = pd.read_csv("data/apollo/formatted_data.csv")
data6 = pd.read_csv('data/emory/input_file_one_outcome.csv')
# data7 = pd.read_csv('data/colombia/formatted_breast_density_project_db.csv')

In [54]:
data_list = [
    (data1, "Chile"), 
    (data2, "UCSF"), 
    (data3, "Taiwan-CMU"), 
    (data4, "Taiwan-CGMH"), 
    (data5, "Apollo"),
    (data6, "Emory")
]

In [55]:
for data, name in data_list:
    data["SITE"] = name
    for col in [
        "Exam Date",
        "Date of Last Negative Mammogram",
        "Date of Cancer Diagnosis",
    ]:
        data[col] = pd.to_datetime(data[col])

In [56]:
data1['Ethnicity'] = "hispanic"
data2['Ethnicity'] = "unk"
data3['Ethnicity'] = "asian"
data4['Ethnicity'] = "asian"
data5['Ethnicity'] = "indian"
data6['Ethnicity'] = data6['Ethnicity'].str.lower()

In [57]:
result = pd.concat([data1, data2, data3, data4, data5, data6], ignore_index=True)

In [58]:
result.to_csv("data/merged_data_2026.csv", index=False)

# Analyze

In [2]:
import numpy as np
from argparse import Namespace
import pandas as pd
from evaluate_density import calculate_metrics, prepare_data, calculate_age_metrics
import datetime
import pickle
from collections import defaultdict, Counter
from scipy._lib._util import rng_integers
from tqdm import tqdm
from typing import NamedTuple
from functools import partial
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import numpy as np
import pandas as pd
import datetime
import argparse
from collections import defaultdict
from tqdm import tqdm
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import statsmodels.formula.api as smf
from joblib import Parallel, delayed
import gc


In [3]:
def ci_bounds(mean, deltas, confidence_interval, num_resamples):
    '''
    Returns the lower and upper bounds of the confidence interval.
    mean : the empirical mean
    deltas : deltas from the empirical mean from different samples
    '''
    deltas = np.sort(deltas)
    index_offset = int((1 - confidence_interval)/2. * num_resamples)
    lower_delta, upper_delta = deltas[index_offset], deltas[-index_offset]
    lower_bound, upper_bound = mean - upper_delta, mean - lower_delta
    return lower_bound, upper_bound

def sample_two_stage_clustered_bootstrap(cluster_ids, n_bootstrap=1000, random_state=None):
    """
    Efficient two-stage clustered bootstrap using only one for loop (over bootstrap replicates).

    Parameters:
    - cluster_ids: 1D array-like with same length as data
    - n_bootstrap: number of bootstrap replicates
    - random_state: int or Generator

    Returns:
    - sampled_indices_list: list of arrays, each containing the resampled indices for a bootstrap replicate
    """
    rng = np.random.default_rng(random_state)
    cluster_ids = np.asarray(cluster_ids)

    # Map unique clusters to data indices
    unique_clusters, inverse_indices = np.unique(cluster_ids, return_inverse=True)
    n_clusters = len(unique_clusters)

    # Preallocate the array for sampled cluster IDs
    sampled_cluster_ids = rng_integers(rng, 0, n_clusters, (n_bootstrap, n_clusters)) # shape (n_bootstrap, n_clusters)

    # Create a list of index arrays, one per cluster
    cluster_to_indices = [np.flatnonzero(inverse_indices == i) for i in range(n_clusters)]

    resampled_indices_list = []
    for b in tqdm(range(n_bootstrap), ncols=100, desc="Bootstrap samples", leave=False):
        # Sample clusters with replacement
        sampled_cluster_ids_step = sampled_cluster_ids[b]

        # Vectorized within-cluster resampling
        resampled_indices = np.concatenate([
            cluster_to_indices[cid][rng.integers(0, len(cluster_to_indices[cid]), size=len(cluster_to_indices[cid]))] 
            if len(cluster_to_indices[cid]) > 1  else cluster_to_indices[cid]  for cid in sampled_cluster_ids_step 
        ]) # shape (n_samples,)
        resampled_indices_list.append(resampled_indices)
    
    return resampled_indices_list

def get_confidence_interval(density, cancer_label, censor_time, patient_ids, confidence_level=0.95, n_bootstrap=1000, random_state=0, statistic_func = None):
    indices = sample_two_stage_clustered_bootstrap(patient_ids, n_bootstrap=n_bootstrap, random_state=random_state)
    positive_always_sampled = sum( cancer_label.iloc[i].sum() > 0 for i in indices)
    retry = 0
    while  (positive_always_sampled < n_bootstrap) and (retry < 100):
        new_indices = sample_two_stage_clustered_bootstrap(
            patient_ids, 
            cancer_label, 
            n_bootstrap= n_bootstrap // 10, 
            random_state=random_state if random_state is None else random_state + 1
        )
        indices = indices + new_indices
        positive_always_sampled += sum( cancer_label.iloc[i].sum() > 0 for i in new_indices)
        retry += 1

    indices = [i for i in indices if cancer_label.iloc[i].sum() > 0][:n_bootstrap]

    
    if statistic_func == "age":
        metrics = [ calculate_age_metrics(density[i], cancer_label.iloc[i], censor_time[i], [70])[0]  for i in indices]
        emp_mean = calculate_age_metrics(density, cancer_label, censor_time, [70])[0]
    else:
        metrics = [ calculate_metrics(density[i], cancer_label.iloc[i], censor_time[i], density_cutoff=3)  for i in indices]
        emp_mean = calculate_metrics(density, cancer_label, censor_time, 3)

    stats = {}
    for metric_name in ['relative_risk', 'odds_ratio', 'c_index']:
        metric_bootstrap = np.array([getattr(m, metric_name) for m in metrics])
        metric_mean = getattr(emp_mean, metric_name)
        deltas = metric_bootstrap - metric_mean
        lower_bound, upper_bound = ci_bounds(metric_mean, deltas, confidence_level, n_bootstrap)
        stats[metric_name] = (metric_mean, lower_bound, upper_bound)
    return stats


In [12]:
def format_data(data: pd.DataFrame, max_followup: int) -> tuple:
    """
    Prepare the data for the evaluation.

    Parameters
    ----------
    data : pd.DataFrame
        The input data
    max_followup : int
        The truncation time in years

    Returns
    -------
    tuple
        A tuple of density, cancer_label, and censor_time
    """
    # set cancer date to 100 if no cancer
    data.loc[data["Cancer (YES | NO)"] == 0, "Date of Cancer Diagnosis"] = data.loc[
        data["Cancer (YES | NO)"] == 0, "Exam Date"
    ] + datetime.timedelta(days=int(100 * 365))

    ages = data["Age at Exam"]
    ethnicity = data["Ethnicity"]
    density = np.array(data["Density"])
    date = data["Exam Date"]
    ever_has_cancer = data["Cancer (YES | NO)"]
    last_negative_date = data["Date of Last Negative Mammogram"]
    cancer_date = data["Date of Cancer Diagnosis"]
    # compute the time of censoring relative to exam date
    years_to_last_negative = np.minimum(
        (last_negative_date - date).dt.days // 365, max_followup
    )
    days_to_last_negative = (last_negative_date - date).dt.days

    # compute the time of cancer diagnosis relative to exam date or max_followup
    years_to_cancer = ((cancer_date - date).dt.days // 365).copy()

    # positives are those who developed cancer within the follow-up period
    cancer_label = (years_to_cancer >= 0) & (years_to_cancer <= max_followup) & ever_has_cancer

    # valid rows
    valid_rows = (
        (years_to_last_negative >= max_followup) & (days_to_last_negative > 0)
    ) | (cancer_label & (years_to_cancer >= 0))

    # construct censor_time array
    censor_time = np.where(cancer_label, years_to_cancer, years_to_last_negative)

    # remove invalid rows
    density = density[valid_rows]
    cancer_label = cancer_label[valid_rows]
    censor_time = censor_time[valid_rows]
    patient_id = data["Patient ID"][valid_rows]
    ages = ages[valid_rows]
    ethnicity = ethnicity[valid_rows]
    site = (
        data["SITE"][valid_rows]
        if "SITE" in data.columns
        else pd.Series([None] * valid_rows.sum(), index=data.index[valid_rows])
    )
    return density, cancer_label, censor_time, patient_id, ages, ethnicity, site

In [2]:
import pandas as pd

In [5]:
data1 = pd.read_csv("data/chile/formatted_complete_DH_data.csv")
data2 = pd.read_csv("data/UCSF/formatted_anonymized_risk_model_preds_w_outcomes_jan_5_2025.csv")
# data3 = pd.read_excel("data/taiwan-cmu/input_ori_20241127.xlsx")
# data4 = pd.read_csv("data/taiwan-cgmh/formatted_MIRAI20250110 to Peter.csv")
data3 = pd.read_excel("data/taiwan-cmu/input_ori_20250808.xlsx")
data4 = pd.read_csv("data/taiwan-cgmh/formatted_minus_asymptomatic_08122025.csv")
data5 = pd.read_csv("data/apollo/formatted_data.csv")
data6 = pd.read_csv('data/emory/input_file_one_outcome.csv')
# data7 = pd.read_csv('data/colombia/formatted_breast_density_project_db.csv')

In [6]:
data_list = [
    (data1, "Chile"),
    (data2, "UCSF"),
    (data3, "Taiwan-CMU"),
    (data4, "Taiwan-CGMH"),
    (data5, "Apollo"),
    (data6, "Emory"),
]

In [22]:
for data, site in data_list:
    for col in [
        "Exam Date",
        "Date of Last Negative Mammogram",
        "Date of Cancer Diagnosis",
    ]:
        data[col] = pd.to_datetime(data[col])

In [3]:
data = pd.read_csv('data/all_sites_minus_colombia_revised.csv')

/tmp/ipykernel_3769676/2577344637.py:1: DtypeWarning: Columns (1,5,7,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('data/all_sites_minus_colombia_revised.csv')


In [4]:
data_list = data.groupby("SITE").apply(lambda x: (x, x.name)).tolist()

/tmp/ipykernel_3769676/2359564751.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_list = data.groupby("SITE").apply(lambda x: (x, x.name)).tolist()


In [28]:
for col in [
    "Exam Date",
    "Date of Last Negative Mammogram",
    "Date of Cancer Diagnosis",
]:
    data[col] = pd.to_datetime(data[col], format = 'ISO8601')

In [18]:
len(data)

961241

In [10]:
len(data1)

440802

In [17]:
density, cancer_label, censor_time, patient_id, ages, ethnicity, site = format_data(data[data['SITE'] == 'Chile'], 4)

In [18]:
len(density)

10308

In [9]:
data[data['SITE'] == 'Chile']

,Exam Date,Patient ID,Age at Exam,Density,birad,Cancer (YES | NO),Date of Cancer Diagnosis,Exam ID,Date of Last Negative Mammogram,Ethnicity,SITE,years_to_cancer,years_to_last_followup,Unnamed: 0,Cancer (YES | NO) OLD,Date of Last Negative Mammogram OLD,Prior Cancer,year
0,2019-10-10,294253,44.0,2.0,0.0,False,2024-10-01,294253_2019-10-10,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2019-10-07,262,53.0,3.0,0.0,False,2024-10-01,262_2019-10-07,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019-10-21,108532,52.0,2.0,0.0,False,2024-10-01,108532_2019-10-21,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2019-10-29,521,54.0,3.0,0.0,False,2024-10-01,521_2019-10-29,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2019-12-02,1043,55.0,3.0,2.0,False,2024-10-01,1043_2019-12-02,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
440797,2024-05-31,278927,54.0,3.0,2.0,False,2024-10-01,278927_2024-05-31,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN
440798,2024-05-31,191327,37.0,3.0,2.0,False,2024-10-01,191327_2024-05-31,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN
440799,2024-05-31,299273,52.0,3.0,0.0,False,2024-10-01,299273_2024-05-31,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN
440800,2024-05-31,61969,46.0,3.0,1.0,False,2024-10-01,61969_2024-05-31,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
data_c = data[data['SITE'] == 'Chile']

In [32]:
data_c[data_c["Patient ID"] == 294253]

,Exam Date,Patient ID,Age at Exam,Density,birad,Cancer (YES | NO),Date of Cancer Diagnosis,Exam ID,Date of Last Negative Mammogram,Ethnicity,SITE,years_to_cancer,years_to_last_followup,Unnamed: 0,Cancer (YES | NO) OLD,Date of Last Negative Mammogram OLD,Prior Cancer,year
0,2019-10-10,294253,44.0,2.0,0.0,False,2024-10-01,294253_2019-10-10,2024-10-01,hispanic,Chile,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
for col in [
    "Exam Date",
    "Date of Last Negative Mammogram",
    "Date of Cancer Diagnosis",
]:
    data[col] = pd.to_datetime(data[col], format = 'ISO8601')

In [8]:
density, cancer_label, censor_time, patient_id, ages, ethnicity, site = format_data(data, 4)

In [16]:
len(data)

961241

In [9]:
minidf = pd.DataFrame(
        {
            "density": density,
            "cancer_outcome": cancer_label.astype(int),
            "time_to_event": censor_time,
            "age": ages,
            "pids": patient_id,
            "ethnicity": ethnicity,
            "site": site
        }
    )

In [12]:
base_formula = "cancer_outcome ~ age + C(site)"
rhs_formula  = "age + C(site)"

In [13]:
cph_full = CoxPHFitter().fit(
        minidf, "time_to_event", "cancer_outcome", formula=f"{rhs_formula} + density"
    )

In [14]:
len(minidf)

262957

In [ ]:
data_list_ = []
for data, site in data_list:

    # 1. Sort by ID and Date (Earliest date first)
    df = data.sort_values(['Patient ID', 'Exam Date'], ascending=[True, True]) 

    # 2. Keep the first occurrence of each ID
    df_first = df.drop_duplicates(subset='Patient ID', keep='first')

    data_list_.append((df_first, site))

In [135]:
density, cancer_label, censor_time, patient_id, ages, ethnicity, site = format_data(data2, 0)

In [136]:
minidf = pd.DataFrame(
        {
            "density": density,
            "cancer_outcome": cancer_label.astype(int),
            "time_to_event": censor_time,
            "age": ages,
            "pids": patient_id,
            "ethnicity": ethnicity,
        }
    )

In [118]:
base_formula = "cancer_outcome ~ age"
rhs_formula  = "age"

In [119]:
cph_full = CoxPHFitter().fit(
        minidf, "time_to_event", "cancer_outcome", formula=f"{rhs_formula} + density"
    )

In [140]:
minidf.loc[:,"site"] = np.array(site)
base_formula = "cancer_outcome ~ age + C(site)"
rhs_formula  = "age + C(site)"

In [141]:
cph_full = CoxPHFitter().fit(
        minidf, "time_to_event", "cancer_outcome", formula=f"{rhs_formula} + density"
    )

In [145]:
model_age  = smf.logit(base_formula, data=minidf).fit(disp=0)
model_full = smf.logit(f"{base_formula} + density", data=minidf).fit(disp=0)

minidf.loc[:,"pred_age"]  = model_age.predict(minidf)
minidf.loc[:,"pred_full"] = model_full.predict(minidf)

# predict and score
c_age  = concordance_index(minidf['time_to_event'], -minidf['pred_age'],  minidf['cancer_outcome'])
c_full = concordance_index(minidf['time_to_event'], -minidf['pred_full'], minidf['cancer_outcome'])


In [ ]:
for group in ["30-40", "40-50", "50-60", "60-70", "70-80"]:

    groupdf = (
        minidf[
            (minidf["age"] >= int(group.split("-")[0]))
            & (minidf["age"] < int(group.split("-")[1]))
        ]
        .copy()
        .reset_index(drop=True)
    )
    if groupdf["cancer_outcome"].nunique() < 2:
        continue

    c_age = concordance_index(
        groupdf["time_to_event"],
        -groupdf["pred_age"],
        groupdf["cancer_outcome"],
    )
    c_full = concordance_index(
        groupdf["time_to_event"],
        -groupdf["pred_full"],
        groupdf["cancer_outcome"],
    )
    delta_c_empirical = c_full - c_age

    

In [147]:
summary_full = cph_full.summary
hr = summary_full.loc["density", "exp(coef)"]
hr_lower = summary_full.loc["density", "exp(coef) lower 95%"]
hr_upper = summary_full.loc["density", "exp(coef) upper 95%"]


In [148]:
for group in ["30-40", "40-50", "50-60", "60-70", "70-80"]:
    groupdf = (
        minidf[
            (minidf["age"] >= int(group.split("-")[0]))
            & (minidf["age"] < int(group.split("-")[1]))
        ]
        .copy()
        .reset_index(drop=True)
    )
    if groupdf["cancer_outcome"].nunique() < 2:
        continue

    cph_full_group = CoxPHFitter().fit(
        groupdf,
        "time_to_event",
        "cancer_outcome",
        formula=f"{rhs_formula} + density",
    )

In [150]:
for group in minidf["ethnicity"].unique():
    groupdf = minidf[minidf["ethnicity"] == group].copy().reset_index(drop=True)
    if groupdf["cancer_outcome"].nunique() < 2:
        continue
    cph_full_group = CoxPHFitter().fit(
        groupdf,
        "time_to_event",
        "cancer_outcome",
        formula=f"{rhs_formula} + density",
    )
    summary_full_group = cph_full_group.summary
    hr_group = summary_full_group.loc["density", "exp(coef)"]
    hr_lower_group = summary_full_group.loc["density", "exp(coef) lower 95%"]
    hr_upper_group = summary_full_group.loc["density", "exp(coef) upper 95%"]
    logit_model_group = smf.logit(
        f"{base_formula} + density", data=groupdf
    ).fit(disp=0)

In [151]:
cph_full.print_summary()

<lifelines.CoxPHFitter: fitted with 867063 total observations, 859565 right-censored observations>
             duration col = 'time_to_event'
                event col = 'cancer_outcome'
      baseline estimation = breslow
   number of observations = 867063
number of events observed = 7498
   partial log-likelihood = -101269.32
         time fit was run = 2026-04-10 19:19:42 UTC

---
                        coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                               
age                     0.04      1.04      0.00            0.04            0.05                1.04                1.05
C(site)[T.Chile]       -1.25      0.29      0.08           -1.40           -1.10                0.25                0.33
C(site)[T.Emory]       -0.86      0.43      0.08           -1.00           -0.71                0.37                0.49
C(site)[T.Taiwan-CGMH] -1.25      0.29      0.14           -1.52           -0.99                0.22                0.37
C(site)[T.Taiwan-CMU]  -1.00      0.37      0.09           -1.18           -0.83                0.31                0.44
C(site)[T.UCSF]        -0.83      0.43      0.08           -0.99           -0.67                0.37                0.51
density                 0.27      1.31      0.02            0.24            0.30                1.27                1.35

                        cmp to      z      p  -log2(p)
covariate                                             
age                       0.00  38.24 <0.005       inf
C(site)[T.Chile]          0.00 -16.59 <0.005    202.92
C(site)[T.Emory]          0.00 -11.32 <0.005     96.27
C(site)[T.Taiwan-CGMH]    0.00  -9.21 <0.005     64.76
C(site)[T.Taiwan-CMU]     0.00 -11.14 <0.005     93.40
C(site)[T.UCSF]           0.00 -10.19 <0.005     78.57
density                   0.00  16.72 <0.005    206.09
---
Concordance = 0.66
Partial AIC = 202552.65
log-likelihood ratio test = 2434.65 on 7 df
-log2(p) of ll-ratio test = inf

## Data Statistics

In [4]:
data_list = [
    (data1, "Chile"), 
    (data2, "UCSF"), 
    (data3, "Taiwan-CMU"), 
    (data4, "Taiwan-CGMH"), 
    (data5, "Apollo"),
    (data6, "Emory"),
    # (data7, "Colombia"),
]

In [5]:
for data, site in data_list:
    print( site, len(data['Exam ID']), len(set( data['Exam ID'])))

Chile 440802 440802
UCSF 97810 97810
Taiwan-CMU 42195 42195
Taiwan-CGMH 13139 13139
Apollo 18768 18768
Emory 348527 348527


In [ ]:
for data, site in data_list:
    for col in [
        "Exam Date",
        "Date of Last Negative Mammogram",
        "Date of Cancer Diagnosis",
    ]:
        data[col] = pd.to_datetime(data[col])
    
    fup_or_cancer = data[ ((data["Date of Last Negative Mammogram"] - data["Exam Date"]).dt.days > 0) | (data['Cancer (YES | NO)'] == 1)]
    num_exams = len(fup_or_cancer)
    num_patients = len(set(fup_or_cancer["Patient ID"]))
    num_cancer = len(fup_or_cancer[(fup_or_cancer['Cancer (YES | NO)'] == 1) ])
    num_cancer_patients = len(set(fup_or_cancer[fup_or_cancer['Cancer (YES | NO)'] == 1]["Patient ID"]))
    print(f"{site}: {data['Exam Date'].min().year}-{data['Exam Date'].max().year}; {num_exams} exams ({num_patients} patients), {num_cancer} cancers ({num_cancer_patients} patients)")
    one_yr_ca = (fup_or_cancer["Date of Cancer Diagnosis"] - fup_or_cancer["Exam Date"]).dt.days < 365
    one_yr_ca_exams = len(fup_or_cancer[(fup_or_cancer['Cancer (YES | NO)'] == 1) & one_yr_ca])
    print(f"\t Cancer prevalence (pct of num patients): {one_yr_ca_exams / num_exams * 100} %")
    

Chile: 2019-2024; 440802 exams (345345 patients), 3108 cancers (2712 patients)
	 Cancer prevalence (pct of num patients): 0.592102576667075 %
UCSF: 2006-2023; 66540 exams (26333 patients), 2978 cancers (1278 patients)
	 Cancer prevalence (pct of num patients): 1.1316501352569883 %
Taiwan-CMU: 2008-2019; 42054 exams (27484 patients), 1199 cancers (860 patients)
	 Cancer prevalence (pct of num patients): 0.8607980215912874 %
Taiwan-CGMH: 2010-2011; 12383 exams (12383 patients), 280 cancers (280 patients)
	 Cancer prevalence (pct of num patients): 0.7671808124041024 %
Apollo: 2014-2024; 10797 exams (7973 patients), 179 cancers (140 patients)
	 Cancer prevalence (pct of num patients): 1.213299990738168 %
Emory: 2013-2024; 294858 exams (55341 patients), 11197 cancers (1768 patients)
	 Cancer prevalence (pct of num patients): 1.189386077366054 %


In [ ]:
minitable=defaultdict(list)
for data, site in data_list:
    fup_or_cancer = data[ ((data["Date of Last Negative Mammogram"] - data["Exam Date"]).dt.days > 0) | (data['Cancer (YES | NO)'] == 1)]
    density_grp = fup_or_cancer.groupby('Density')
    for name, group in density_grp:
        minitable["site"].append(site)
        minitable["density"].append(name)
        minitable["exams"].append(len(group['Exam ID'].unique()))
        minitable["patients"].append(len(group['Patient ID'].unique()))
pd.DataFrame(minitable)

,site,density,exams,patients
0,Chile,1.0,63056,53004
1,Chile,2.0,238073,192178
2,Chile,3.0,126449,104321
3,Chile,4.0,13224,11578
4,UCSF,1.0,8939,4291
5,UCSF,2.0,31076,12732
6,UCSF,3.0,18561,10125
7,UCSF,4.0,7964,3751
8,Taiwan-CMU,1.0,1107,917
9,Taiwan-CMU,2.0,5912,4846


In [ ]:
minitable = pd.DataFrame(minitable)

In [ ]:
minitable[minitable['density'] > 2].groupby('site')['exams'].sum() / minitable.groupby('site')['exams'].sum()

site
Apollo         0.346485
Chile          0.316861
Emory          0.482002
Taiwan-CGMH    0.623678
Taiwan-CMU     0.833096
UCSF           0.398632
Name: exams, dtype: float64

In [ ]:
minitable['exams'].sum()

np.int64(867434)

In [ ]:
minitable=defaultdict(list)
for data, site in data_list:
    fup_or_cancer = data[ ((data["Date of Last Negative Mammogram"] - data["Exam Date"]).dt.days > 0) | (data['Cancer (YES | NO)'] == 1)]
    if "Ethnicity" not in fup_or_cancer.columns:
        continue
    race_grp = fup_or_cancer.groupby('Ethnicity')
    for name, group in race_grp:
        minitable["site"].append(site)
        minitable["ethnicity"].append(name)
        minitable["exams"].append(len(group['Exam ID'].unique()))
        minitable["patients"].append(len(group['Patient ID'].unique()))
pd.DataFrame(minitable)

,site,ethnicity,exams,patients
0,Chile,Chilean,440802,345345
1,UCSF,UCSF,66540,26333
2,Taiwan-CMU,1,42054,27484
3,Apollo,Region_1,5000,3701
4,Apollo,Region_2,4673,3391
5,Apollo,Region_3,1124,881
6,Emory,Asian,16613,3671
7,Emory,Black,137485,25074
8,Emory,White,140760,26596


In [ ]:
minitable=defaultdict(list)
for data, site in data_list:
    # Define bins and labels
    bins = [30, 40, 50, 60, 70, 80]
    labels = ['30-40', '40-50', '50-60', '60-70', '70-80']
    # Bin the 'value' column
    data['range'] = pd.cut(data['Age at Exam'], bins=bins, labels=labels, right=False)
    fup_or_cancer = data[ ((data["Date of Last Negative Mammogram"] - data["Exam Date"]).dt.days > 0) | (data['Cancer (YES | NO)'] == 1)]
    age_grp = fup_or_cancer.groupby('range')
    for name, group in age_grp:
        minitable["site"].append(site)
        minitable["range"].append(name)
        minitable["exams"].append(len(group['Exam ID'].unique()))
        minitable["patients"].append(len(group['Patient ID'].unique()))
pd.DataFrame(minitable)

/tmp/ipykernel_1737736/1934696815.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_grp = fup_or_cancer.groupby('range')
/tmp/ipykernel_1737736/1934696815.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_grp = fup_or_cancer.groupby('range')
/tmp/ipykernel_1737736/1934696815.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_grp = fup_or_cancer.groupby('range')
/tmp/ipykernel_1737736/1934696815.py:9: Fut

,site,range,exams,patients
0,Chile,30-40,13863,12678
1,Chile,40-50,104788,87523
2,Chile,50-60,178926,141570
3,Chile,60-70,119505,97077
4,Chile,70-80,21063,17925
5,UCSF,30-40,1046,745
6,UCSF,40-50,14710,8125
7,UCSF,50-60,19025,9834
8,UCSF,60-70,18472,8730
9,UCSF,70-80,11020,4925


In [ ]:
minitable=defaultdict(list)
for finterval in [0,4]:
    for data, site in data_list:
        fup = data[ ((data["Date of Last Negative Mammogram"] - data["Exam Date"]).dt.days > 0 ) & ((data["Date of Last Negative Mammogram"] - data["Exam Date"]).dt.days // 365 >= finterval) & (data['Cancer (YES | NO)'] == 0)]
        cancer = data[ ((data["Date of Cancer Diagnosis"] - data["Exam Date"]).dt.days // 365  < finterval+1) & (data['Cancer (YES | NO)'] == 1)]
        minitable["site"].append(site)
        minitable["site"].append(site)
        minitable["interval"].append(finterval+1)
        minitable["interval"].append(finterval+1)
        minitable["group"].append('fup')
        minitable["exams"].append(len(fup['Exam ID'].unique()))
        minitable["patients"].append(len(fup['Patient ID'].unique()))
        minitable["group"].append('cancer')
        minitable["exams"].append(len(cancer['Exam ID'].unique()))
        minitable["patients"].append(len(cancer['Patient ID'].unique()))
pd.DataFrame(minitable)

,site,interval,group,exams,patients
0,Chile,1,fup,437694,342633
1,Chile,1,cancer,2610,2545
2,UCSF,1,fup,63562,25055
3,UCSF,1,cancer,753,751
4,Taiwan-CMU,1,fup,40855,26624
5,Taiwan-CMU,1,cancer,362,362
6,Taiwan-CGMH,1,fup,12103,12103
7,Taiwan-CGMH,1,cancer,95,95
8,Apollo,1,fup,10618,7879
9,Apollo,1,cancer,131,127


10308

In [ ]:
for finterval in [0,4]:
    print(f"Percent of all exams preceding a diagnosis within {finterval+1} year")
    for data, site in data_list:
        negative = (((data["Date of Last Negative Mammogram"] - data["Exam Date"]).dt.days > 0 ) & ( (data["Date of Last Negative Mammogram"] - data["Exam Date"]).dt.days // 365 >= finterval) & (data['Cancer (YES | NO)'] == 0))
        positive = ((data["Date of Cancer Diagnosis"] - data["Exam Date"]).dt.days <= 365*(finterval+1) ) & (data['Cancer (YES | NO)'] == 1)
        exams = data[ negative | positive ]
        cancer = data[ positive ]
        print(site, len(cancer), len(exams), len(cancer['Exam ID'].unique()) / len(data['Exam ID'].unique()) * 100 )
    print()

Percent of all exams preceding a diagnosis within 1 year
Chile 2611 440305 0.5923294358918516
UCSF 1029 64591 1.0520396687455271
Taiwan-CMU 363 41218 0.8602915037326698
Taiwan-CGMH 95 12198 0.7230382829743511
Apollo 131 10749 0.697996589940324
Emory 3508 287169 1.0065217328930038

Percent of all exams preceding a diagnosis within 5 year
Chile 3108 10308 0.7050784706058503
UCSF 2068 38021 2.114303240977405
Taiwan-CMU 843 39260 1.997867045858514
Taiwan-CGMH 207 10221 1.5754623639546388
Apollo 177 2196 0.9430946291560102
Emory 8884 159411 2.549013419333366



3508

In [ ]:
print(f"Percent of all exams that are dense")
for data, site in data_list:
    dense_exams = data[data['Density'] > 2]
    print(site, len(dense_exams['Exam ID'].unique()) / len(data['Exam ID'].unique()) * 100 )

Percent of all exams that are dense
Chile 31.686108502230027
UCSF 43.87076985993252
Taiwan-CMU 83.33214835881029
Taiwan-CGMH 62.45528579039501
Apollo 36.51960784313725
Emory 48.179911455927375


In [ ]:
data_list = [
    (data3, "Taiwan-CMU"), 
    (data4, "Taiwan-CGMH"), 
]

## Metrics

In [ ]:
for data, site in data_list:
    for col in [
        "Exam Date",
        "Date of Last Negative Mammogram",
        "Date of Cancer Diagnosis",
    ]:
        data[col] = pd.to_datetime(data[col])

In [ ]:
minitable = defaultdict(list)
for max_followup in [0,4]:
    print(f"{max_followup+1}-year cancer")
    for data, site in data_list:
        density, cancer_label, censor_time, patient_ids, patient_age = format_data(data, max_followup, include_age=True)
        # metrics = calculate_metrics(density, cancer_label, censor_time, 3)
        # print(f"{site} \t {metrics.relative_risk} \t {metrics.odds_ratio} \t {metrics.c_index}")
        metrics = get_confidence_interval(density, cancer_label, censor_time, patient_ids, confidence_level=0.95, n_bootstrap=10, random_state=None)
        minitable["site"].append(site)
        minitable["max_followup"].append(max_followup+1)
        minitable["relative_risk"].append(f"{metrics['relative_risk'][0]:.2f} ({metrics['relative_risk'][1]:.2f}-{metrics['relative_risk'][2]:.2f})")
        minitable["odds_ratio"].append(f"{metrics['odds_ratio'][0]:.2f} ({metrics['odds_ratio'][1]:.2f}-{metrics['odds_ratio'][2]:.2f})")
        minitable["c_index"].append(f"{metrics['c_index'][0]:.2f} ({metrics['c_index'][1]:.2f}-{metrics['c_index'][2]:.2f})")
pd.DataFrame(minitable)

1-year cancer


5-year cancer


,site,max_followup,relative_risk,odds_ratio,c_index
0,Chile,1,1.39 (1.40-1.40),1.39 (1.40-1.40),0.55 (0.55-0.55)
1,UCSF,1,1.10 (1.20-1.20),1.10 (1.20-1.20),0.52 (0.54-0.54)
2,Taiwan-CMU,1,1.34 (1.63-1.63),1.35 (1.64-1.64),0.52 (0.54-0.54)
3,Taiwan-CGMH,1,1.08 (1.27-1.27),1.08 (1.28-1.28),0.53 (0.55-0.55)
4,Apollo,1,1.54 (2.08-2.08),1.55 (2.10-2.10),0.54 (0.61-0.61)
5,Emory,1,1.03 (1.12-1.12),1.03 (1.13-1.13),0.51 (0.52-0.52)
6,Chile,5,1.15 (1.18-1.18),1.22 (1.27-1.27),0.54 (0.55-0.55)
7,UCSF,5,1.43 (1.56-1.56),1.46 (1.60-1.60),0.56 (0.57-0.57)
8,Taiwan-CMU,5,1.39 (1.62-1.62),1.40 (1.63-1.63),0.52 (0.53-0.53)
9,Taiwan-CGMH,5,1.20 (1.45-1.45),1.21 (1.46-1.46),0.52 (0.55-0.55)


In [ ]:
pd.DataFrame(minitable)

,site,max_followup,relative_risk,odds_ratio,c_index
0,Taiwan-CMU,1,1.34 (0.70-1.68),1.35 (0.70-1.69),0.52 (0.49-0.54)
1,Taiwan-CGMH,1,1.08 (0.49-1.45),1.08 (0.48-1.45),0.53 (0.48-0.58)
2,Taiwan-CMU,5,1.39 (0.98-1.65),1.40 (0.98-1.67),0.52 (0.50-0.54)
3,Taiwan-CGMH,5,1.20 (0.81-1.51),1.21 (0.81-1.52),0.52 (0.49-0.56)


In [ ]:
print()

In [ ]:
# minitable = defaultdict(list)
# for max_followup in [0,4]:
#     for data, site in data_list:
#         for col in [
#             "Exam Date",
#             "Date of Last Negative Mammogram",
#             "Date of Cancer Diagnosis",
#         ]:
#             data[col] = pd.to_datetime(data[col])
#         for group in ["30-40", "40-50", "50-60", "60-70", "70-80"]:
#             try:
#                 age_lower, age_upper = group.split("-")
#                 age_lower, age_upper = int(age_lower), int(age_upper)
#                 group_data = data[
#                     (data["Age at Exam"] >= age_lower) & (data["Age at Exam"] < age_upper)
#                 ]
#                 if len(group_data) == 0:
#                     continue
#                 density, cancer_label, censor_time, patient_ids = format_data(group_data, max_followup)
#                 # metrics = calculate_metrics(
#                 #     density,
#                 #     cancer_label,
#                 #     censor_time,
#                 #     3,
#                 # )
#                 metrics = get_confidence_interval(density, cancer_label, censor_time, patient_ids, confidence_level=0.95, n_bootstrap=1000, random_state=None)
#                 minitable["Site"].append(site)
#                 minitable["Group"].append("Age: " + str(age_lower) + "-" + str(age_upper))
#                 minitable["Censoring Year"].append(max_followup)
#                 minitable["Relative Risk"].append(f"{metrics['relative_risk'][0]:.2f} ({metrics['relative_risk'][1]:.2f}-{metrics['relative_risk'][2]:.2f})")
#                 minitable["Odds Ratio"].append(f"{metrics['odds_ratio'][0]:.2f} ({metrics['odds_ratio'][1]:.2f}-{metrics['odds_ratio'][2]:.2f})")
#                 minitable["C-Index"].append(f"{metrics['c_index'][0]:.2f} ({metrics['c_index'][1]:.2f}-{metrics['c_index'][2]:.2f})")
#             except:
#                 continue          

In [ ]:
# pd.DataFrame(minitable)

In [ ]:
# age >= 70
minitable = defaultdict(list)
for max_followup in [0,4]:
    print(f"{max_followup+1}-year cancer")
    for data, site in data_list:
        for col in [
            "Exam Date",
            "Date of Last Negative Mammogram",
            "Date of Cancer Diagnosis",
        ]:
            data[col] = pd.to_datetime(data[col])
        density, cancer_label, censor_time, patient_ids, ages = format_data(data, max_followup, include_age=True)
        # metrics = calculate_age_metrics(ages, cancer_label, censor_time, [70])[0]
        # print(f"{site} \t {round(metrics.relative_risk,2)} \t {round(metrics.odds_ratio,2)} \t {round(metrics.c_index, 2)}")
        ages = np.array(ages)
        metrics = get_confidence_interval(ages, cancer_label, censor_time, patient_ids, confidence_level=0.95, n_bootstrap=1000, random_state=None, statistic_func="age")
        minitable["site"].append(site)
        minitable["max_followup"].append(max_followup+1)
        minitable["Relative Risk"].append(f"{metrics['relative_risk'][0]:.2f} ({metrics['relative_risk'][1]:.2f}-{metrics['relative_risk'][2]:.2f})")
        minitable["Odds Ratio"].append(f"{metrics['odds_ratio'][0]:.2f} ({metrics['odds_ratio'][1]:.2f}-{metrics['odds_ratio'][2]:.2f})")
        minitable["C-Index"].append(f"{metrics['c_index'][0]:.2f} ({metrics['c_index'][1]:.2f}-{metrics['c_index'][2]:.2f})")

1-year cancer


5-year cancer


In [ ]:
pd.DataFrame(minitable)

,site,max_followup,Relative Risk,Odds Ratio,C-Index
0,Taiwan-CMU,1,0.00 (0.00-0.00),0.00 (0.00-0.00),0.50 (0.50-0.50)
1,Taiwan-CGMH,1,0.00 (0.00-0.00),0.00 (0.00-0.00),0.50 (0.50-0.50)
2,Taiwan-CMU,5,1.40 (-1.14-2.81),1.42 (-1.38-2.83),0.50 (0.50-0.50)
3,Taiwan-CGMH,5,0.00 (0.00-0.00),0.00 (0.00-0.00),0.50 (0.50-0.50)


## Adjusted for Age

In [ ]:
import statsmodels.formula.api as smf
from joblib import Parallel, delayed

from sklearn.linear_model import LogisticRegression
from lifelines.utils import concordance_index


In [ ]:
def run_single_bootstrap(i, data, p_indices, u_patients, n_p):
    # Set seed based on iteration for reproducibility in parallel
    np.random.seed(i)
    
    # Fast clustered sampling
    sample_pids = np.random.choice(u_patients, size=n_p, replace=True)
    indices = [idx for pid in sample_pids for idx in p_indices[pid]]
    sample_df = data.iloc[indices]
    
    try:
        c_age = CoxPHFitter().fit(sample_df, 'time_to_event', 'cancer_outcome', formula="age").concordance_index_
        c_full = CoxPHFitter().fit(sample_df, 'time_to_event', 'cancer_outcome', formula="age + density").concordance_index_
        return (c_age, c_full, c_full - c_age)
    except:
        return None


def calculate_age_adjusted_metrics(density, cancer_label, censor_time, ages, pids, confidence_level=0.95, n_bootstrap=1000, random_state=0):

    minidf = pd.DataFrame({
        'density': density,
        'cancer_outcome': cancer_label.astype(int),
        'time_to_event': censor_time,
        'age': ages,
        "pids": pids
    })

    grouped_indices = minidf.groupby("pids").indices
    unique_patients = minidf["pids"].unique()
    n_patients = len(unique_patients)

    n_bootstrap = 1000
    results = Parallel(n_jobs=-1)(
        delayed(run_single_bootstrap)(i, minidf, grouped_indices, unique_patients, n_patients) 
        for i in tqdm(range(n_bootstrap))
    )

    unique_patients = minidf['pids'].unique()
    n_patients = len(unique_patients)
    # Faster sampling for large data
    grouped_indices = minidf.groupby('pids').indices

    # CPH for c-index
    c_ages= []
    c_deltas = [] 
    c_fulls = []
    iterations = 0
    
    with tqdm(ncols=100, total=2000, leave=False) as tqdm_bar:
        while len(c_deltas) < n_bootstrap:
            tqdm_bar.update(1)
            iterations += 1
            # 1. Sample Patient IDs with replacement
            sample_ids = np.random.choice(unique_patients, size=n_patients, replace=True)
            
            # 2. Collect all exams for the chosen IDs
            # Note: If a patient is picked twice, they appear twice in the sample 
            # (with different temporary IDs to avoid index conflicts)
            all_row_indices = [idx for pid in sample_ids for idx in grouped_indices[pid]]
            sample_df = minidf.iloc[all_row_indices]
    
            if sample_df['cancer_outcome'].sum() == 0:
                continue
    
            cph_age = CoxPHFitter().fit(sample_df, 'time_to_event', 'cancer_outcome', formula="age")
            c_age = cph_age.concordance_index_
            
            cph_full = CoxPHFitter().fit(sample_df, 'time_to_event', 'cancer_outcome', formula="age + density")
            c_full = cph_full.concordance_index_
            
            delta_c = c_full - c_age
            c_ages.append(c_age)
            c_fulls.append(c_full)
            c_deltas.append(delta_c)
            if iterations > 2000:
                break 

    # Cox proportional hazards model for ratio 
    cph_age = CoxPHFitter().fit(minidf, 'time_to_event', 'cancer_outcome', formula="age", cluster_col="pids")
    c_age = cph_age.concordance_index_
    
    cph_full = CoxPHFitter().fit(minidf, 'time_to_event', 'cancer_outcome', formula="age + density", cluster_col="pids")
    c_full = cph_full.concordance_index_
    
    delta_c = c_full - c_age

    # CI for c-index
    deltas = np.array(c_deltas) -  delta_c
    delta_lower, delta_upper = ci_bounds(delta_c, deltas, confidence_level, n_bootstrap)


    # adjusted hazard ratio for density
    summary_full = cph_full.summary
    hr = summary_full.loc['density', 'exp(coef)']
    hr_lower = summary_full.loc['density', 'exp(coef) lower 95%']
    hr_upper = summary_full.loc['density', 'exp(coef) upper 95%']

    # Logistic regression for the 3-year binary outcome
    logit_model = smf.logit("cancer_outcome ~ age + density", data=minidf).fit(disp=0)

    results = {
        "full_model_cindex": c_full,
        "age_only_model_cindex": c_age,
        "delta_c_index": delta_c,
        "delta_c_index_ci": (delta_lower, delta_upper),
        "density_adjusted_hr": hr,
        "density_adjusted_or": np.exp(logit_model.params['density']),
        "density_adjusted_hr_ci": (hr_lower, hr_upper),
        "density_adjusted_or_ci": logit_model.conf_int().loc['density'].apply(np.exp).tolist(),
    }
    return results

In [ ]:
import statsmodels.formula.api as smf
from joblib import Parallel, delayed

def run_single_bootstrap_log(i, data, p_indices, u_patients, n_p):
    # Set seed based on iteration for reproducibility in parallel
    np.random.seed(i)
    
    # Fast clustered sampling
    sample_pids = np.random.choice(u_patients, size=n_p, replace=True)
    # indices = np.array([idx for pid in sample_pids for idx in p_indices[pid]])
    # sample_df = data.iloc[indices]
    pid_counts = Counter(sample_pids)
    sample_df = data[data['pids'].isin(pid_counts)].copy()
    sample_df['weight'] = sample_df['pids'].map(pid_counts)
    w = sample_df['weight'].values
    try:
        X = sample_df[['age', 'density']].values
        X_age = sample_df[['age']].values
        y = sample_df['cancer_outcome'].values
        t = sample_df['time_to_event'].values
        
        # probs_age  = LogisticRegression().fit(X_age, y, sample_weight=w).predict_proba(X_age)[:, 1]
        # probs_full = LogisticRegression().fit(X, y, sample_weight=w).predict_proba(X)[:, 1]

        # c_age  = concordance_index(t, -probs_age,  y)
        # c_full = concordance_index(t, -probs_full, y)

        model_age  = smf.logit("cancer_outcome ~ age", data=sample_df).fit(disp=0)
        model_full = smf.logit("cancer_outcome ~ age + density", data=sample_df).fit(disp=0)

        # predict and score
        c_age  = concordance_index(sample_df['time_to_event'], -model_age.predict(sample_df),  sample_df['cancer_outcome'])
        c_full = concordance_index(sample_df['time_to_event'], -model_full.predict(sample_df), sample_df['cancer_outcome'])
                
        
        return (c_age, c_full, c_full - c_age)
    except:
        return None    
    
def run_single_bootstrap(i, data, p_indices, u_patients, n_p):
    # Set seed based on iteration for reproducibility in parallel
    np.random.seed(i)
    
    # Fast clustered sampling
    sample_pids = np.random.choice(u_patients, size=n_p, replace=True)
    indices = [idx for pid in sample_pids for idx in p_indices[pid]]
    sample_df = data.iloc[indices]
    
    try:
        c_age = CoxPHFitter().fit(sample_df, 'time_to_event', 'cancer_outcome', formula="age").concordance_index_
        c_full = CoxPHFitter().fit(sample_df, 'time_to_event', 'cancer_outcome', formula="age + density").concordance_index_
        return (c_age, c_full, c_full - c_age)
    except:
        return None


def calculate_age_adjusted_metrics(density, cancer_label, censor_time, ages, pids, confidence_level=0.95, n_bootstrap=1000, random_state=0):

    minidf = pd.DataFrame({
        'density': density,
        'cancer_outcome': cancer_label.astype(int),
        'time_to_event': censor_time,
        'age': ages,
        "pids": pids
    })
    grouped_indices = minidf.groupby('pids').indices
    unique_patients = minidf['pids'].unique()
    n_patients = len(unique_patients)
    
    results = Parallel(n_jobs=-2)(
        delayed(run_single_bootstrap_log)(i, minidf, grouped_indices, unique_patients, n_patients) 
        for i in tqdm(range(n_bootstrap), ncols=100)
    )

    # Logistic regression for the binary outcome
    logit_age  = smf.logit("cancer_outcome ~ age", data=minidf).fit(disp=0)
    logit_model = smf.logit("cancer_outcome ~ age + density", data=minidf).fit(disp=0)
    # predict and score
    c_age  = concordance_index(minidf['time_to_event'], -logit_age.predict(minidf),  minidf['cancer_outcome'])
    c_full = concordance_index(minidf['time_to_event'], -logit_model.predict(minidf), minidf['cancer_outcome'])
                


    # # CPH for c-index
    c_ages= [c[0]for c in results if c is not None]
    c_deltas = [c[2] for c in results if c is not None ]
    c_fulls = [c[1] for c in results if c is not None]

    # print('computing mean')
    # # Cox proportional hazards model for ratio 
    # cph_age = CoxPHFitter().fit(minidf, 'time_to_event', 'cancer_outcome', formula="age")
    # c_age = cph_age.concordance_index_
    
    cph_full = CoxPHFitter().fit(minidf, 'time_to_event', 'cancer_outcome', formula="age + density")
    # c_full = cph_full.concordance_index_

    delta_c = c_full - c_age

    # CI for c-index
    deltas = np.array(c_deltas) -  delta_c
    delta_lower, delta_upper = ci_bounds(delta_c, deltas, confidence_level, n_bootstrap)

    # adjusted hazard ratio for density
    summary_full = cph_full.summary
    hr = summary_full.loc['density', 'exp(coef)']
    hr_lower = summary_full.loc['density', 'exp(coef) lower 95%']
    hr_upper = summary_full.loc['density', 'exp(coef) upper 95%']

    results = {
        "full_model_cindex": c_full,
        "age_only_model_cindex": c_age,
        "delta_c_index": delta_c,
        "delta_c_index_ci": (delta_lower, delta_upper),
        "density_adjusted_hr": hr,
        "density_adjusted_or": np.exp(logit_model.params['density']),
        "density_adjusted_hr_ci": (hr_lower, hr_upper),
        "density_adjusted_or_ci": logit_model.conf_int().loc['density'].apply(np.exp).tolist(),
    }
    return results

In [29]:
for data, site in data_list:
    for col in [
        "Exam Date",
        "Date of Last Negative Mammogram",
        "Date of Cancer Diagnosis",
    ]:
        data[col] = pd.to_datetime(data[col])

In [32]:
data_list = [(data6, 'emory')]
minitable = defaultdict(list)
for max_followup in [0,4]:
    print(f"{max_followup+1}-year cancer")
    for data, site in tqdm(data_list, ncols=100):
        density, cancer_label, censor_time, patient_ids, patient_age = format_data(data, max_followup, include_age=True)
        metrics = calculate_age_adjusted_metrics(density, cancer_label, censor_time, patient_age, patient_ids, n_bootstrap=10)
        minitable["site"].append(site)
        minitable["max_followup"].append(max_followup+1)
        minitable["density_c_index"].append(f"{metrics['delta_c_index']:.2f} ({metrics['delta_c_index_ci'][0]:.2f}-{metrics['delta_c_index_ci'][1]:.2f})")
        minitable["hazard_ratio"].append(f"{metrics['density_adjusted_hr']:.2f} ({metrics['density_adjusted_hr_ci'][0]:.2f}-{metrics['density_adjusted_hr_ci'][1]:.2f})")
        minitable["odds_ratio"].append(f"{metrics['density_adjusted_or']:.2f} ({metrics['density_adjusted_or_ci'][0]:.2f}-{metrics['density_adjusted_or_ci'][1]:.2f})")
        
pd.DataFrame(minitable)

1-year cancer


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:18<00:00, 18.34s/it]


5-year cancer


100%|█████████████████████████████████████████████████████████████████| 1/1 [00:10<00:00, 10.71s/it]


,site,max_followup,density_c_index,hazard_ratio,odds_ratio
0,emory,1,0.01 (0.01-0.01),1.21 (1.15-1.27),1.21 (1.15-1.27)
1,emory,5,0.01 (0.01-0.01),1.24 (1.20-1.27),1.25 (1.21-1.29)


### MGH

In [ ]:
new_mgh_and_merged_results = pickle.load(open("data/mgh_and_merged_results.p", "rb"))

In [ ]:
non_age_indices = [i for i, g in enumerate(new_mgh_and_merged_results[0]['Group']) if '-Age:' not in g]
non_age_followups = new_mgh_and_merged_results[0]['Followup']
age_indices = [i for i, g in enumerate(new_mgh_and_merged_results[0]['Group']) if '-Age:' in g]
age_followups = [i for i in range(0,10) for _ in range(10)]
followups = np.array([-1 for _ in range(len(new_mgh_and_merged_results[0]['Group']))])
np.put(followups, non_age_indices, non_age_followups)
np.put(followups, age_indices, age_followups)
new_mgh_and_merged_results[0]['Followup'] = followups.tolist()

In [ ]:
mgh_results = pd.DataFrame(new_mgh_and_merged_results[0])

In [ ]:
(mgh_results['Censoring Year'] == mgh_results['Followup'] ).all()

True

In [ ]:
new_mgh_and_merged_results[1]

{'Dates': (Timestamp('2009-01-02 00:00:00'), Timestamp('2023-01-09 00:00:00')),
 'Num exams': 642173,
 'Num patients': 134646,
 'Num cancer': 16666,
 'Num cancer pids': 5701,
 'density1 exams': 49847,
 'density1 patients': 14683,
 'density2 exams': 325273,
 'density2 patients': 81329,
 'density3 exams': 241900,
 'density3 patients': 65925,
 'density4 exams': 25153,
 'density4 patients': 9581,
 'Asian exams': 32817,
 'Asian patients': 7620,
 'Black exams': 29610,
 'Black patients': 6646,
 'Native Hawaiian or Other Pacific Islander exams': 172,
 'Native Hawaiian or Other Pacific Islander patients': 32,
 'Other exams': 28319,
 'Other patients': 6929,
 'Two or More exams': 2692,
 'Two or More patients': 628,
 'White exams': 526706,
 'White patients': 107849,
 'hispanic exams': 9703,
 'hispanic patients': 2240,
 'other exams': 501,
 'other patients': 125,
 'unk exams': 11653,
 'unk patients': 2621,
 '30-40 exams': 3516,
 '30-40 patients': 3249,
 '40-50 exams': 142626,
 '40-50 patients': 494

In [ ]:
mgh_results[mgh_results['Group'] == 'MGH-All']

,Group,Censoring Year,Relative Risk,Odds Ratio,C-Index,Followup
18,MGH-All,0,1.165067,1.166473,0.525417,0
53,MGH-All,1,1.153597,1.156009,0.523835,1
88,MGH-All,2,1.129285,1.13263,0.521818,2
123,MGH-All,3,1.090159,1.093531,0.517471,3
158,MGH-All,4,1.059274,1.06227,0.513846,4
192,MGH-All,5,1.029406,1.03113,0.510209,5
226,MGH-All,6,1.00175,1.00187,0.506510,6
260,MGH-All,7,0.970989,0.968602,0.501906,7
294,MGH-All,8,0.938456,0.932297,0.496214,8
328,MGH-All,9,0.911546,0.900462,0.490723,9


In [ ]:
mgh_results[mgh_results['Group'] == 'MGH-Age as Predictor: 70']

,Group,Censoring Year,Relative Risk,Odds Ratio,C-Index,Followup
28,MGH-Age as Predictor: 70,0,1.751724,1.760699,0.552851,0
58,MGH-Age as Predictor: 70,1,1.757795,1.774648,0.553581,1
88,MGH-Age as Predictor: 70,2,1.904895,1.940994,0.560304,2
118,MGH-Age as Predictor: 70,3,2.042512,2.108401,0.565398,3
148,MGH-Age as Predictor: 70,4,2.184897,2.295951,0.569810,4
177,MGH-Age as Predictor: 70,5,2.430886,2.607309,0.577654,5
205,MGH-Age as Predictor: 70,6,2.710931,2.995444,0.585424,6
231,MGH-Age as Predictor: 70,7,2.99857,3.457131,0.592637,7
257,MGH-Age as Predictor: 70,8,3.26891,4.007,0.599260,8
283,MGH-Age as Predictor: 70,9,3.490576,4.681645,0.605433,9


## Combined

In [ ]:
mgh_results[mgh_results['Group'] == 'Merged-All']

,Group,Censoring Year,Relative Risk,Odds Ratio,C-Index,Followup
0,Merged-All,0,1.229571,1.231632,0.533509,0
30,Merged-All,1,1.208478,1.211794,0.531765,1
60,Merged-All,2,1.137465,1.141023,0.524306,2
90,Merged-All,3,1.069323,1.071988,0.515930,3
120,Merged-All,4,0.991893,0.991454,0.505455,4
150,Merged-All,5,0.979059,0.977639,0.503327,5
179,Merged-All,6,0.982336,0.980816,0.503875,6
207,Merged-All,7,0.984114,0.982353,0.504064,7
233,Merged-All,8,0.969701,0.965314,0.501456,8
259,Merged-All,9,0.960382,0.952468,0.499471,9


In [ ]:
mgh_results[mgh_results['Group'] == 'Merged-Age as Predictor: 70']


,Group,Censoring Year,Relative Risk,Odds Ratio,C-Index,Followup
18,Merged-Age as Predictor: 70,0,2.045714,2.060494,0.560329,0
48,Merged-Age as Predictor: 70,1,2.108732,2.137488,0.563952,1
78,Merged-Age as Predictor: 70,2,2.12993,2.180138,0.565078,2
108,Merged-Age as Predictor: 70,3,2.134969,2.213707,0.565314,3
138,Merged-Age as Predictor: 70,4,2.130121,2.247194,0.565318,4
167,Merged-Age as Predictor: 70,5,2.314715,2.503579,0.571224,5
195,Merged-Age as Predictor: 70,6,2.515064,2.82178,0.577079,6
221,Merged-Age as Predictor: 70,7,2.704188,3.202309,0.582608,7
247,Merged-Age as Predictor: 70,8,2.860549,3.674097,0.587920,8
273,Merged-Age as Predictor: 70,9,2.905955,4.221902,0.592957,9


In [ ]:
mgh_results['Group'].unique()

array(['Merged-All', 'Merged-white', 'Merged-black', 'Merged-hispanic',
       'Merged-asian', 'Merged-indian', 'Merged-unk', 'Merged-other',
       'Merged-Age: 30-40', 'Merged-Age: 40-50', 'Merged-Age: 50-60',
       'Merged-Age: 60-70', 'Merged-Age: 70-80',
       'Merged-Age as Predictor: 40', 'Merged-Age as Predictor: 50',
       'Merged-Age as Predictor: 60', 'Merged-Age as Predictor: 70',
       'Merged-Age as Predictor: All ages', 'MGH-All', 'MGH-white',
       'MGH-black', 'MGH-hispanic', 'MGH-asian', 'MGH-unk', 'MGH-other',
       'MGH-Age: 30-40', 'MGH-Age: 40-50', 'MGH-Age: 50-60',
       'MGH-Age: 60-70', 'MGH-Age: 70-80', 'MGH-Age as Predictor: 40',
       'MGH-Age as Predictor: 50', 'MGH-Age as Predictor: 60',
       'MGH-Age as Predictor: 70', 'MGH-Age as Predictor: All ages'],
      dtype=object)

In [ ]:
mgh_results[mgh_results['Group'] == 'Merged-Age: 70-80']

,Group,Censoring Year,Relative Risk,Odds Ratio,C-Index,Followup
12,Merged-Age: 70-80,0,1.213291,1.216604,0.531314,0
47,Merged-Age: 70-80,1,1.21273,1.218623,0.532817,1
82,Merged-Age: 70-80,2,1.172086,1.179826,0.530841,2
117,Merged-Age: 70-80,3,1.141659,1.151166,0.529659,3
152,Merged-Age: 70-80,4,1.131856,1.144567,0.530169,4
186,Merged-Age: 70-80,5,1.137707,1.155351,0.531814,5
220,Merged-Age: 70-80,6,1.142241,1.166575,0.533789,6
254,Merged-Age: 70-80,7,1.138444,1.17024,0.534323,7
288,Merged-Age: 70-80,8,1.120471,1.157649,0.532867,8
322,Merged-Age: 70-80,9,1.084655,1.119543,0.529084,9


## Minus MGH

In [ ]:
result = pd.concat([d[0] for d in data_list], ignore_index=True)

In [ ]:
print(f"Rows: {len(result)} \t Exams: {len(result['Exam ID'])} \t Patients: {len(result['Patient ID'].unique())}")

Rows: 1012271 	 Exams: 1012271 	 Patients: 490688


In [ ]:
num_cancer_pts = result.groupby('Patient ID')['Cancer (YES | NO)'].max().sum()
print(f"Ca Exams: {result['Cancer (YES | NO)'].sum()} \t Ca Patients: {num_cancer_pts}")

Ca Exams: 21068.0 	 Ca Patients: 7414.0


In [ ]:
for col in [
    "Exam Date",
    "Date of Last Negative Mammogram",
    "Date of Cancer Diagnosis",
]:
    result[col] = pd.to_datetime(result[col])

In [ ]:
# use age as a predictor
result = pd.concat([data1, data2, data3], ignore_index=True)
for col in [
    "Exam Date",
    "Date of Last Negative Mammogram",
    "Date of Cancer Diagnosis",
]:
    result[col] = pd.to_datetime(result[col])

for max_followup in [0,4]:
    print(f"{max_followup+1}-year cancer")
    density, cancer_label, censor_time, ages, _ = format_data(result, max_followup, include_age=True)
    metrics = calculate_age_metrics(ages, cancer_label, censor_time, [65, 70])
    for m, age in zip(metrics, [65, 70]):
        print(f"age: {age}")
        print(f"{m.relative_risk} \t {m.odds_ratio} \t {m.c_index}")
    
    

1-year cancer
age: 65
1.9486963796029286 	 1.9595343143060107 	 0.5615094685647004
age: 70
2.4931315429978094 	 2.516093133168347 	 0.5504981216364171
5-year cancer
age: 65
1.6673113686700463 	 1.7358698343032748 	 0.5509523406700918
age: 70
2.1016833956429037 	 2.253726962565278 	 0.5449203963624147
